# 🍜 STEP 02 — 단일 에이전트: ChatCompletionAgent

**STEP 01**에서 만든 Plugin을 `ChatCompletionAgent`에 장착해
F&B 전문 에이전트 3종을 각각 단독으로 구현합니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 |
|------|------|
| `ChatCompletionAgent` | 역할·지시문·플러그인이 있는 전문 에이전트 |
| `agent.get_response()` | 단일 응답 요청 |
| `agent.invoke_stream()` | 스트리밍 응답 (실시간 출력) |
| `AgentThread` | 에이전트별 대화 상태 유지 |

---

## 이 단계에서 만들 에이전트
```
┌─────────────────────┐  ┌──────────────────────┐  ┌────────────────────────┐
│   LegalTaxAgent     │  │   LocationAgent      │  │   FinanceAgent         │
│ 법령·세무 안내       │  │ 상권 분석            │  │ 재무 리스크 분석       │
│ FnbLegalPlugin      │  │ FnbLocationPlugin    │  │ FnbFinancePlugin       │
└─────────────────────┘  └──────────────────────┘  └────────────────────────┘
```

In [1]:
import asyncio, os, json
from typing import Annotated
from dotenv import load_dotenv

from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.contents import ChatHistory

load_dotenv()
print("임포트 완료")

임포트 완료


---
## 1. 헬퍼: Kernel 팩토리 함수

> **왜 에이전트마다 별도 Kernel을 만드나?**
> 
> 각 에이전트에게 다른 플러그인 세트만 노출시키기 위해서입니다.
> LegalTaxAgent는 법령 플러그인만, LocationAgent는 지도 플러그인만 갖습니다.

In [2]:
def make_kernel() -> Kernel:
    """기본 Azure OpenAI 서비스가 등록된 빈 Kernel을 반환합니다."""
    kernel = Kernel()
    kernel.add_service(
        AzureChatCompletion(
            deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
            endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-03-01-preview"),
        )
    )
    return kernel

def make_auto_settings():
    s = AzureChatPromptExecutionSettings()
    s.function_choice_behavior = FunctionChoiceBehavior.Auto()
    return s

print("헬퍼 함수 정의 완료")

헬퍼 함수 정의 완료


---
## 2. 플러그인 정의 (STEP 01 복습 + FinancePlugin 추가)

In [3]:
# ─────────────────────────────────────────
# Legal/Tax Plugin
# Azure RAI filter blocks Korean in tool descriptions
# → All descriptions and Annotated params use English
# ─────────────────────────────────────────
class FnbLegalPlugin:
    @kernel_function(
        description="Returns F&B business license requirements and required documents by business type."
    )
    def get_license_requirements(
        self,
        business_type: Annotated[str, "Business type (e.g. general restaurant, cafe)"],
        area_sqm: Annotated[float, "Business area in square meters"],
    ) -> str:
        docs = {
            "일반음식점": ["영업신고(보건소)", "위생교육(6h)", "소방완비증명(300㎡↑)"],
            "카페":       ["영업신고(보건소)", "위생교육(3h)"],
        }
        items = docs.get(business_type, ["해당 정보 없음"])
        if business_type == "일반음식점" and area_sqm >= 300:
            items.append("소방완비증명서 필수")
        return "\n".join(items)

    @kernel_function(
        description="Returns recommended tax type (simplified or general) based on expected annual revenue."
    )
    def get_tax_type_guide(
        self,
        annual_revenue_krw: Annotated[int, "Expected annual revenue in KRW"],
    ) -> str:
        if annual_revenue_krw < 104_000_000:
            return "간이과세자 대상 (연 매출 1억 400만원 미만) — 부가세 부담 낮으나 세금계산서 발행 불가"
        return "일반과세자 대상 — 부가세 10%, 세금계산서 발행 가능"


# ─────────────────────────────────────────
# Location/Commercial Area Plugin
# ─────────────────────────────────────────
class FnbLocationPlugin:
    @kernel_function(
        description="Returns commercial area analysis (foot traffic, competitors, rent) for a given location and business type."
    )
    def analyze_commercial_area(
        self,
        location: Annotated[str, "Location to analyze (e.g. Hongdae, Mapo-gu, Seoul)"],
        business_type: Annotated[str, "Business type to analyze"],
    ) -> str:
        # Mock — real implementation: Kakao Local API
        return json.dumps(
            {
                "location": location,
                "daily_floating_population": 45000,
                "competitor_count_500m": 12,
                "avg_monthly_rent_krw_per_sqm": 85000,
                "commercial_rating": "A",
            },
            ensure_ascii=False,
        )


# ─────────────────────────────────────────
# Finance / BEP Plugin
# ─────────────────────────────────────────
class FnbFinancePlugin:
    @kernel_function(
        description="Calculates break-even point using initial investment, fixed costs, avg revenue per customer, and variable cost ratio."
    )
    def calculate_bep(
        self,
        initial_investment_krw:      Annotated[int,   "Total initial investment amount in KRW"],
        monthly_fixed_cost_krw:      Annotated[int,   "Monthly fixed costs (rent, labor, etc.) in KRW"],
        avg_revenue_per_customer_krw:Annotated[int,   "Average revenue per customer in KRW"],
        variable_cost_ratio:         Annotated[float, "Variable cost ratio between 0 and 1 (e.g. 0.35 for 35%)"],
    ) -> str:
        cm   = 1.0 - variable_cost_ratio
        bep_revenue   = monthly_fixed_cost_krw / cm
        bep_customers = bep_revenue / avg_revenue_per_customer_krw
        profit_at_bep_plus20 = bep_revenue * 0.2 * cm
        payback = initial_investment_krw / profit_at_bep_plus20 if profit_at_bep_plus20 > 0 else float('inf')
        return (
            "monthly_bep_revenue: %s KRW, "
            "monthly_bep_customers: %d persons, "
            "daily_bep_customers: %d persons, "
            "payback_months: %.1f months"
            % ("{:,.0f}".format(bep_revenue), int(bep_customers), int(bep_customers/30), payback)
        )


print("플러그인 3종 정의 완료")


플러그인 3종 정의 완료


---
## 3. 전문 에이전트 3종 생성

> **`ChatCompletionAgent` 구조**:
> ```
> ChatCompletionAgent(
>     name         = 에이전트 이름 (GroupChat에서 식별자)
>     description  = 이 에이전트가 무엇을 하는지 (트리아지 에이전트가 라우팅 참고)
>     instructions = 시스템 프롬프트 (역할 + 행동 규칙)
>     kernel       = Plugin이 등록된 Kernel
>     arguments    = 실행 설정 (FunctionChoiceBehavior 등)
> )
> ```

In [4]:
# ── Agent 1: Legal/Tax ──
legal_kernel = make_kernel()
legal_kernel.add_plugin(FnbLegalPlugin(), plugin_name="FnbLegal")

legal_tax_agent = ChatCompletionAgent(
    name="LegalTaxAgent",
    description="F&B startup legal, license, and tax specialist agent",
    instructions=(
        "You are an F&B startup legal and tax expert. "
        "Provide accurate information based on Korean Food Sanitation Act, "
        "Commercial Building Lease Protection Act, Labor Standards Act, and VAT Act. "
        "Always cite the relevant legal article (e.g. Food Sanitation Act Article 37). "
        "For uncertain legal interpretations, recommend consulting the relevant authority. "
        "Always respond in Korean."
    ),
    kernel=legal_kernel,
    arguments=KernelArguments(settings=make_auto_settings()),
)

# ── Agent 2: Location/Commercial Area ──
location_kernel = make_kernel()
location_kernel.add_plugin(FnbLocationPlugin(), plugin_name="FnbLocation")

location_agent = ChatCompletionAgent(
    name="LocationAgent",
    description="Commercial area analysis specialist using foot traffic, competitor data, and Kakao Maps",
    instructions=(
        "You are an F&B startup commercial area analysis expert. "
        "Analyze using Kakao Maps API and SEMAS (Small Enterprise and Market Service) data. "
        "Provide foot traffic, competitor count, and rent level with concrete numbers. "
        "Structure your analysis as Risk / Opportunity factors. "
        "Always respond in Korean."
    ),
    kernel=location_kernel,
    arguments=KernelArguments(settings=make_auto_settings()),
)

# ── Agent 3: Finance / BEP ──
finance_kernel = make_kernel()
finance_kernel.add_plugin(FnbFinancePlugin(), plugin_name="FnbFinance")

finance_agent = ChatCompletionAgent(
    name="FinanceAgent",
    description="BEP calculation, scenario analysis, and risk simulation specialist",
    instructions=(
        "You are an F&B startup finance analysis expert. "
        "Provide BEP calculation, scenario-based revenue forecast, and risk assessment with numbers. "
        "Always include optimistic, base, and pessimistic scenarios. "
        "Format numbers with commas (e.g. 1,500,000 KRW). "
        "Always respond in Korean."
    ),
    kernel=finance_kernel,
    arguments=KernelArguments(settings=make_auto_settings()),
)

print("에이전트 3종 생성 완료!")
print("- LegalTaxAgent:", legal_tax_agent.name)
print("- LocationAgent:", location_agent.name)
print("- FinanceAgent: ", finance_agent.name)


에이전트 3종 생성 완료!
- LegalTaxAgent: LegalTaxAgent
- LocationAgent: LocationAgent
- FinanceAgent:  FinanceAgent


---
## 4. 단일 응답 테스트 — `get_response()`

> 한 번의 질문에 한 번의 응답을 받는 가장 단순한 패턴입니다.

In [5]:
# LegalTaxAgent test
# ⚠️ Azure RAI filter blocks Korean user messages → use English questions
response = await legal_tax_agent.get_response(
    messages=(
        "I am planning to open a 30-pyeong (99 sqm) general restaurant in Gangnam-gu, Seoul. "
        "Please explain the required license and permit procedures. Respond in Korean."
    )
)
print("[LegalTaxAgent 응답]")
print(response.content)


[LegalTaxAgent 응답]
일반음식점을 운영하기 위해서는 사업자등록, 식품영업허가, 위생교육 수료 등이 필요합니다. 자세한 서류와 절차는 해당 구청 또는 관련 기관에서 확인하실 수 있습니다. 정확한 정보를 위해 강남구청에 문의하시기 바랍니다. 식품위생법 제37조에 따라 영업 허가 및 신고 절차가 적용되므로 이를 참고하세요.


In [6]:
# LocationAgent test
response2 = await location_agent.get_response(
    messages=(
        "I am considering opening a cafe near Hongik University Station in Mapo-gu, Seoul. "
        "Please provide a commercial area analysis. Respond in Korean."
    )
)
print("[LocationAgent 응답]")
print(response2.content)


[LocationAgent 응답]
홍대입구역 인근에서 카페를 여는 것에 대한 상권 분석을 아래와 같이 제공해드립니다.

### 리스크 요인
1. **경쟁자 수**: 500m 반경 내에 총 12개의 경쟁 카페가 존재합니다. 이는 경쟁 강도가 매우 높음을 의미하며, 차별화된 전략이 필요합니다.

2. **임대료**: 평균 월 임대료는 제곱미터당 85,000원입니다. 이 지역의 임대료 수준이 비교적 높은 편이므로, 초기 투자 비용 및 운영 비용을 신중히 고려해야 합니다.

### 기회 요인
1. **유동 인구**: 일일 유동 인구가 약 45,000명에 달하며, 이는 많은 잠재 고객을 확보할 수 있는 좋은 기회를 제공합니다.

2. **상권 등급**: 해당 지역의 상권 등급은 'A'로 평가되며, 이는 비즈니스 운영에 유리한 환경을 의미합니다.

종합적으로 홍대입구역 인근은 높은 유동 인구를 활용할 수 있는 매력적인 위치이지만, 높은 경쟁과 임대료를 극복하기 위한 차별화된 전략이 필요합니다.


In [7]:
# FinanceAgent test
response3 = await finance_agent.get_response(
    messages=(
        "I am planning to open a cafe. "
        "Initial investment: 80,000,000 KRW, "
        "Monthly fixed cost: 3,500,000 KRW, "
        "Average revenue per customer: 7,000 KRW, "
        "Variable cost ratio: 35%. "
        "Please calculate the break-even point. Respond in Korean."
    )
)
print("[FinanceAgent 응답]")
print(response3.content)


[FinanceAgent 응답]
카페의 손익분기점 계산 결과는 다음과 같습니다:

- **월간 손익분기점 수익**: 5,384,615 KRW
- **월간 손익분기점 고객 수**: 769 명
- **일일 손익분기점 고객 수**: 25 명
- **투자 회수 기간**: 약 114.3 개월

이 정보를 기반으로 운영 계획을 세우시면 됩니다. 추가로 필요하신 사항이 있으시면 언제든지 말씀해 주세요.


---
## 5. 멀티턴 대화 — `AgentThread` 로 대화 상태 유지

> `AgentThread`는 에이전트와의 대화 세션입니다.
> 같은 thread를 계속 전달하면 이전 대화 내용을 기억합니다.
>
> STEP 01의 핸드오프 오케스트레이터에서 `thread_checkpoint_store`가 바로 이 역할!

In [8]:
from semantic_kernel.agents import ChatHistoryAgentThread

async def multi_turn_demo():
    """FinanceAgent multi-turn conversation — remembers previous context."""
    thread = ChatHistoryAgentThread()

    # ⚠️ Azure RAI filter: all user messages must be in English
    #    Add 'Respond in Korean.' at the end of each question
    questions = [
        (
            "Calculate BEP with initial investment 80,000,000 KRW, "
            "monthly fixed cost 3,500,000 KRW, avg revenue per customer 7,000 KRW, "
            "variable cost ratio 35%. Respond in Korean."
        ),
        # 이전 대화 맥락 기억해야 답 가능
        "Based on the BEP just calculated, how many customers per day are needed to break even? Respond in Korean.",
        # 연속 맥락
        "If the average revenue per customer increases to 9,000 KRW, how does the BEP change? Respond in Korean.",
    ]

    for i, q in enumerate(questions, 1):
        print("\n" + "="*60)
        print("[질문 %d]" % i)
        print("-"*60)
        response = await finance_agent.get_response(
            messages=q,
            thread=thread,
        )
        thread = response.thread
        print(response.content)

await multi_turn_demo()



[질문 1]
------------------------------------------------------------
귀하의 F&B 스타트업의 손익분기점(BEP) 계산 결과는 다음과 같습니다:

- **월간 BEP 매출**: 5,384,615 KRW
- **월간 BEP 고객수**: 769명
- **일일 BEP 고객수**: 25명
- **투자 회수 기간**: 약 114.3개월

위의 BEP를 달성하기 위해 하루에 최소 25명의 고객을 확보해야 하며, 매출을 통해 약 9년 6개월 이후에 초기 투자금을 회수할 수 있습니다.

[질문 2]
------------------------------------------------------------
귀하의 F&B 스타트업이 손익분기점을 달성하기 위해 필요한 일일 고객 수는 **25명**입니다.

[질문 3]
------------------------------------------------------------
고객당 평균 매출이 9,000 KRW로 증가할 경우, 손익분기점(BEP)의 변동은 다음과 같습니다:

- **월간 BEP 매출**: 5,384,615 KRW (변동 없음)
- **월간 BEP 고객수**: 598명
- **일일 BEP 고객수**: 19명
- **투자 회수 기간**: 약 114.3개월 (변동 없음)

고객당 매출이 증가함에 따라, 하루에 필요한 고객 수가 **19명**으로 감소하였습니다.


---
## 6. 스트리밍 응답 — `invoke_stream()`

> 실제 서비스에서는 `invoke_stream()`으로 토큰을 실시간 출력합니다.
> FastAPI에서 `StreamingResponse`로 프론트에 전달하는 방식입니다.

In [9]:
async def streaming_demo():
    """LegalTaxAgent streaming response."""
    print("[LegalTaxAgent 스트리밍 응답]")
    print("-" * 50)

    async for chunk in legal_tax_agent.invoke_stream(
        messages=(
            "Please explain 3 key rights that tenants must know under the "
            "Korean Commercial Building Lease Protection Act. Respond in Korean."
        )
    ):
        print(chunk.content, end="", flush=True)

    print()

await streaming_demo()


[LegalTaxAgent 스트리밍 응답]
--------------------------------------------------
상가건물 임대차보호법에 따라 임차인이 알아야 할 주요 권리는 다음과 같습니다:

1. **계약갱신요구권**: 임차인은 임대차 계약기간이 종료되기 전, 계약 갱신을 요구할 수 있으며, 임대인은 정당한 사유 없이 이를 거절하지 못합니다. 이러한 권리는 임대차 보호를 강화하여 임차인의 안정적인 사업 운영을 지원합니다 (상가건물 임대차보호법 제10조).

2. **임대료 증액 제한**: 임대인은 특정 기간 동안 임대료를 무제한으로 증액할 수 없으며, 법에서 정한 일정 비율을 초과하여 증액할 수 없습니다. 이는 임차인을 보호하여 불합리한 임대료 증액을 방지합니다 (상가건물 임대차보호법 제11조).

3. **대항력 및 권리금 보호**: 임차인은 계약 체결과 동시에 관할 지자체에 사업자등록을 하여 대항력을 가질 수 있으며, 임차인은 계약 종료 시 권리금을 회수할 권리를 가집니다. 권리금 회수는 임차인이 장기간에 걸쳐 투자한 비용에 대한 보호를 의미합니다 (상가건물 임대차보호법 제3조 및 제10조의4).

이 법률 관련하여 명확한 해석이 필요할 경우, 법률 전문 변호사와의 상담을 추천합니다.


---
## 7. ✅ STEP 02 정리

| 배운 것 | 코드 패턴 | 다음 단계 연결 |
|---------|-----------|---------------|
| 전문 에이전트 생성 | `ChatCompletionAgent(name, instructions, kernel)` | STEP 03 GroupChat 참여자 |
| 단일 응답 | `agent.get_response(messages)` | 기본 호출 패턴 |
| 멀티턴 대화 | `thread = ChatHistoryAgentThread()` | 세션 관리의 기초 |
| 스트리밍 | `async for chunk in agent.invoke_stream()` | FastAPI StreamingResponse |

---

### 다음 STEP 미리보기
```
STEP 03: AgentGroupChat (멀티 에이전트)
  - 3개 에이전트를 하나의 그룹으로 묶기
  - KernelFunctionSelectionStrategy: 어떤 에이전트가 다음에 응답할지 결정
  - KernelFunctionTerminationStrategy: 언제 대화를 끝낼지 결정
  - 트리아지 에이전트(오케스트레이터) 패턴
```